# BrainIB — Interpretable GNN for ASD Classification
> **ABIDE · PyTorch Geometric · Brain Information Bottleneck · nilearn**

### What this notebook does
1. Installs dependencies  
2. Mounts Google Drive — data goes ONLY to `MyDrive/brainib_data/` (isolated, deletable)  
3. Downloads ABIDE, extracts 116-ROI FC matrices  
4. Trains BrainIB (+ optional GCN baseline)  
5. Evaluates on test set and produces glass-brain overlays

**Runtime**: GPU recommended (Runtime → Change runtime type → T4 GPU)

## Cell 1 — Install dependencies
Run once per Colab session.

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-1000:])
    return result.returncode == 0

# Detect torch version for PyG wheel
import torch
tv = torch.__version__.split('+')[0]   # e.g. '2.1.0'
cv = 'cu121' if torch.cuda.is_available() else 'cpu'
print(f'torch={tv}  cuda={cv}')

# Install PyG
run('pip install torch-geometric -q')
run(f'pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{tv}+{cv}.html -q')

# Install other deps
run('pip install nilearn scikit-learn tqdm -q')
print('Done — restart runtime if prompted, then run from Cell 2 onwards')

## Cell 2 — Mount Drive & set paths
Data lands in `MyDrive/brainib_data/` ONLY.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# Tell configs.py where Drive is
os.environ['DRIVE_ROOT'] = '/content/drive/MyDrive'

# Verify we only write here
DATA_DIR = '/content/drive/MyDrive/brainib_data'
os.makedirs(DATA_DIR, exist_ok=True)
print(f'Data root: {DATA_DIR}')
print('Files here will persist across sessions.')

## Cell 3 — Clone / upload project code
If you uploaded the zip, adjust the path below.

In [ ]:
# Option A: clone from GitHub (if you pushed it there)
# !git clone https://github.com/YOUR_USERNAME/brainib.git /content/brainib

# Option B: copy from Drive if you uploaded it there
# !cp -r '/content/drive/MyDrive/brainib' /content/brainib

# Option C: we inline everything below (no clone needed — cells ARE the code)
import sys, os
PROJECT = '/content/brainib'
os.makedirs(PROJECT, exist_ok=True)
sys.path.insert(0, PROJECT)
print('Project root:', PROJECT)

## Cell 4 — Write all source files inline
Run this once; it writes every module to /content/brainib/

In [ ]:
%%writefile /content/brainib/configs.py
import os
DRIVE_ROOT   = os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive')
DATA_DIR     = os.path.join(DRIVE_ROOT, 'brainib_data')
CACHE_DIR    = os.path.join(DATA_DIR, 'nilearn_cache')
FC_DIR       = os.path.join(DATA_DIR, 'fc_matrices')
CKPT_DIR     = os.path.join(DATA_DIR, 'checkpoints')
FIG_DIR      = os.path.join(DATA_DIR, 'figures')
RESULTS_DIR  = os.path.join(DATA_DIR, 'results')

ABIDE_PIPELINE       = 'cpac'
ABIDE_STRATEGY       = 'filt_global'
ABIDE_DERIVATIVES    = ['func_preproc']
ABIDE_N_SUBJECTS     = None
ATLAS                = 'aal'
N_ROIS               = 116
CONNECTIVITY_KIND    = 'correlation'

EDGE_THRESHOLD_PCTILE = 70
SELF_LOOPS            = False

HIDDEN_DIM   = 64
LATENT_DIM   = 32
NUM_CLASSES  = 2
DROPOUT      = 0.3
GCN_LAYERS   = 3

BETA         = 0.01
GAMMA        = 0.5

EPOCHS       = 100
BATCH_SIZE   = 32
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 20
TRAIN_RATIO  = 0.7
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15
SEED         = 42
TOP_K_EDGES  = 30

In [ ]:
# Write data/__init__.py and models/__init__.py and utils/__init__.py
import os
for pkg in ['data', 'models', 'utils']:
    os.makedirs(f'/content/brainib/{pkg}', exist_ok=True)
    open(f'/content/brainib/{pkg}/__init__.py', 'w').close()
print('Package dirs created')

In [ ]:
%%writefile /content/brainib/data/dataset.py
import os, sys, numpy as np, pandas as pd, torch
from torch_geometric.data import Data, Dataset
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from configs import DATA_DIR, FC_DIR, N_ROIS, EDGE_THRESHOLD_PCTILE, SELF_LOOPS, SEED, TRAIN_RATIO, VAL_RATIO

def fc_to_graph(fc, threshold_pctile=EDGE_THRESHOLD_PCTILE):
    n = fc.shape[0]
    x = torch.tensor(fc, dtype=torch.float32)
    abs_fc = np.abs(fc)
    threshold = np.percentile(abs_fc, threshold_pctile)
    rows, cols = np.where((abs_fc > threshold) & (np.tri(n, k=-1) == 1))
    src = np.concatenate([rows, cols])
    dst = np.concatenate([cols, rows])
    edge_index = torch.tensor(np.stack([src, dst]), dtype=torch.long)
    edge_attr  = torch.tensor(
        np.concatenate([abs_fc[rows, cols], abs_fc[rows, cols]]),
        dtype=torch.float32).unsqueeze(1)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

class ABIDEDataset(Dataset):
    def __init__(self, split='all', transform=None):
        super().__init__(transform=transform)
        manifest_path = os.path.join(DATA_DIR, 'manifest.csv')
        if not os.path.exists(manifest_path):
            raise FileNotFoundError(f'Manifest not found at {manifest_path}. Run download first.')
        df = pd.read_csv(manifest_path)
        df = df.dropna(subset=['fc_path'])
        df = df[df['fc_path'].apply(os.path.exists)]
        rng = np.random.RandomState(SEED)
        idx = rng.permutation(len(df))
        n = len(df); n_train = int(n * TRAIN_RATIO); n_val = int(n * VAL_RATIO)
        if   split == 'train': df = df.iloc[idx[:n_train]]
        elif split == 'val':   df = df.iloc[idx[n_train:n_train+n_val]]
        elif split == 'test':  df = df.iloc[idx[n_train+n_val:]]
        self.records = df.reset_index(drop=True)
        print(f'[dataset] {split:5s} n={len(self.records)} ASD={self.records["label"].sum()} TC={(self.records["label"]==0).sum()}')
    def len(self): return len(self.records)
    def get(self, idx):
        row = self.records.iloc[idx]
        fc  = np.load(row['fc_path'])
        g   = fc_to_graph(fc)
        g.y = torch.tensor([int(row['label'])], dtype=torch.long)
        g.subject_id = str(row['subject_id'])
        g.fc_raw     = torch.tensor(fc, dtype=torch.float32)
        return g

In [ ]:
%%writefile /content/brainib/models/losses.py
import torch, torch.nn.functional as F
from configs import BETA, GAMMA

def kl_divergence(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

def edge_sparsity_loss(edge_mask):
    return edge_mask.mean()

def ib_loss(logits, labels, mu, logvar, edge_mask, beta=BETA, gamma=GAMMA):
    cls      = F.cross_entropy(logits, labels)
    kl       = kl_divergence(mu, logvar)
    sparsity = edge_sparsity_loss(edge_mask)
    total    = cls + beta * kl + gamma * sparsity
    return total, cls, kl, sparsity

In [ ]:
%%writefile /content/brainib/models/brainib.py
import sys, torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from configs import HIDDEN_DIM, LATENT_DIM, NUM_CLASSES, DROPOUT, GCN_LAYERS, N_ROIS

class GCNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        dims = [N_ROIS] + [HIDDEN_DIM]*GCN_LAYERS
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i+1]) for i in range(GCN_LAYERS)])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(HIDDEN_DIM) for _ in range(GCN_LAYERS)])
        self.mu_head    = nn.Linear(HIDDEN_DIM*2, LATENT_DIM)
        self.logvar_head = nn.Linear(HIDDEN_DIM*2, LATENT_DIM)
    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, edge_index)))
            x = F.dropout(x, p=DROPOUT, training=self.training)
        g = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=-1)
        return self.mu_head(g), self.logvar_head(g), x

class EdgeSelector(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(HIDDEN_DIM*2, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, node_emb, edge_index):
        cat   = torch.cat([node_emb[edge_index[0]], node_emb[edge_index[1]]], dim=-1)
        soft  = torch.sigmoid(self.mlp(cat).squeeze(-1))
        hard  = (soft > 0.5).float() - soft.detach() + soft
        return soft, hard

class BrainIB(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder    = GCNEncoder()
        self.selector   = EdgeSelector()
        self.classifier = nn.Sequential(
            nn.Linear(LATENT_DIM, HIDDEN_DIM), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(HIDDEN_DIM, NUM_CLASSES))
    def reparameterise(self, mu, logvar):
        if self.training:
            return mu + (0.5*logvar).exp() * torch.randn_like(mu)
        return mu
    def forward(self, data):
        mu, logvar, node_emb = self.encoder(data.x, data.edge_index, data.batch)
        z = self.reparameterise(mu, logvar)
        soft_mask, hard_mask = self.selector(node_emb, data.edge_index)
        node_w = self._scatter(soft_mask, data.edge_index, data.x.size(0))
        combined = z + global_mean_pool(node_emb * node_w.unsqueeze(-1), data.batch)
        return {'logits': self.classifier(combined), 'mu': mu, 'logvar': logvar,
                'edge_mask': soft_mask, 'hard_mask': hard_mask, 'node_emb': node_emb}
    @staticmethod
    def _scatter(mask, ei, n):
        s = torch.zeros(n, device=mask.device); c = torch.zeros(n, device=mask.device)
        s.scatter_add_(0, ei[0], mask); c.scatter_add_(0, ei[0], torch.ones_like(mask))
        return s / c.clamp(min=1)

In [ ]:
%%writefile /content/brainib/models/gcn_baseline.py
import sys, torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from configs import HIDDEN_DIM, NUM_CLASSES, DROPOUT, GCN_LAYERS, N_ROIS

class VanillaGCN(nn.Module):
    def __init__(self):
        super().__init__()
        dims = [N_ROIS] + [HIDDEN_DIM]*GCN_LAYERS
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i+1]) for i in range(GCN_LAYERS)])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(HIDDEN_DIM) for _ in range(GCN_LAYERS)])
        self.head  = nn.Sequential(nn.Linear(HIDDEN_DIM*2, HIDDEN_DIM), nn.ReLU(),
                                   nn.Dropout(DROPOUT), nn.Linear(HIDDEN_DIM, NUM_CLASSES))
    def forward(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, ei)))
            x = F.dropout(x, p=DROPOUT, training=self.training)
        g = torch.cat([global_mean_pool(x, b), global_max_pool(x, b)], dim=-1)
        return {'logits': self.head(g)}

In [ ]:
%%writefile /content/brainib/utils/metrics.py
import numpy as np, torch
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix

def compute_metrics(labels, probs, preds=None):
    labels = np.array(labels); probs = np.array(probs)
    if preds is None: preds = (probs > 0.5).astype(int)
    acc = accuracy_score(labels, preds)
    try: auc = roc_auc_score(labels, probs)
    except: auc = float('nan')
    report = classification_report(labels, preds, target_names=['TC','ASD'], output_dict=True)
    return {'accuracy': acc, 'auc_roc': auc, 'report': report,
            'confusion_matrix': confusion_matrix(labels, preds).tolist()}

def edge_sparsity(hard_masks):
    masks = torch.cat(hard_masks)
    return (masks < 0.5).float().mean().item()

def print_metrics(m, prefix=''):
    tag = f'[{prefix}] ' if prefix else ''
    print(f'{tag}accuracy : {m["accuracy"]:.4f}')
    print(f'{tag}AUC-ROC  : {m["auc_roc"]:.4f}')
    r = m['report']
    print(f'{tag}ASD F1   : {r["ASD"]["f1-score"]:.4f}  TC F1: {r["TC"]["f1-score"]:.4f}')

In [ ]:
%%writefile /content/brainib/utils/visualize.py
import os, sys, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from configs import FIG_DIR, N_ROIS, TOP_K_EDGES

def _aal_coords():
    from nilearn import datasets as nds
    from nilearn.plotting import find_parcellation_cut_coords
    return find_parcellation_cut_coords(nds.fetch_atlas_aal().maps)

def plot_glass_brain_subgraph(fc_matrix, edge_scores, title='Subgraph',
                               filename='subgraph.png', top_k=TOP_K_EDGES, label=1):
    from nilearn import plotting
    os.makedirs(FIG_DIR, exist_ok=True)
    coords = _aal_coords()
    scores_upper = np.triu(edge_scores, k=1)
    flat = scores_upper.flatten()
    thresh = np.sort(flat)[-top_k] if len(flat) >= top_k else flat.min()
    adj = (scores_upper >= thresh).astype(float) * scores_upper
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    cmap = 'Reds' if label == 1 else 'Blues'
    for ax, view in zip(axes, ['x','y','z']):
        plotting.plot_connectome(adj, coords, node_size=10, edge_cmap=cmap,
            edge_vmin=0, edge_vmax=scores_upper.max()+1e-6,
            display_mode=view, axes=ax, annotate=False, colorbar=(view=='z'))
    label_str = 'ASD' if label==1 else 'TC'
    fig.suptitle(f'{title} — {label_str} (top {top_k} edges)', fontsize=11, y=1.02)
    fig.tight_layout()
    out = os.path.join(FIG_DIR, filename)
    fig.savefig(out, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'[viz] {out}'); return out

def plot_fc_matrix(fc, title='FC matrix', filename='fc.png'):
    os.makedirs(FIG_DIR, exist_ok=True)
    fig, ax = plt.subplots(figsize=(7,6))
    im = ax.imshow(fc, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=11); ax.set_xlabel('ROI'); ax.set_ylabel('ROI')
    fig.tight_layout()
    out = os.path.join(FIG_DIR, filename)
    fig.savefig(out, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'[viz] {out}'); return out

def plot_training_curves(history, filename='training_curves.png'):
    os.makedirs(FIG_DIR, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(12,4))
    axes[0].plot(history['train_loss'], label='train'); axes[0].plot(history['val_loss'], label='val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(history['train_acc'], label='train'); axes[1].plot(history['val_acc'], label='val')
    axes[1].set_title('Accuracy'); axes[1].legend()
    fig.tight_layout()
    out = os.path.join(FIG_DIR, filename)
    fig.savefig(out, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'[viz] {out}'); return out

def aggregate_edge_importance(model_outputs, n_rois=N_ROIS):
    import torch
    score_mat = np.zeros((n_rois, n_rois)); count_mat = np.zeros((n_rois, n_rois))
    for out in model_outputs:
        mask = out['edge_mask'].detach().cpu().numpy()
        ei   = out['edge_index'].cpu().numpy()
        for e, (s, d) in enumerate(ei.T):
            if s < n_rois and d < n_rois:
                score_mat[s, d] += mask[e]; count_mat[s, d] += 1
    return score_mat / np.where(count_mat == 0, 1, count_mat)

## Cell 5 — Download ABIDE & extract FC matrices
> **First time only (~30–60 min). Cached to Drive after.**
> Set `N_SUBJECTS = 50` to test quickly first.

In [ ]:
import os, sys
sys.path.insert(0, '/content/brainib')
os.environ['DRIVE_ROOT'] = '/content/drive/MyDrive'

# ─── CHANGE THIS ─────────────────────────────────────────────────────────
N_SUBJECTS = 50   # set None to get all 871 (takes longer + more Drive space)
# ─────────────────────────────────────────────────────────────────────────

import numpy as np, pandas as pd
from nilearn import datasets as nds
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure
from tqdm.notebook import tqdm
from configs import DATA_DIR, CACHE_DIR, FC_DIR

for d in [DATA_DIR, CACHE_DIR, FC_DIR]: os.makedirs(d, exist_ok=True)

print('Fetching ABIDE metadata...')
abide = nds.fetch_abide_pcp(
    data_dir=CACHE_DIR, pipeline='cpac',
    band_pass_filtering=True, global_signal_regression=False,
    derivatives=['func_preproc'], n_subjects=N_SUBJECTS, verbose=0)
print(f'Got {len(abide.func_preproc)} subjects')

atlas   = nds.fetch_atlas_aal()
masker  = NiftiLabelsMasker(labels_img=atlas.maps, standardize=True,
                            memory=CACHE_DIR, verbose=0)
conn    = ConnectivityMeasure(kind='correlation')
records = []

for func_path, label, sid in tqdm(
        zip(abide.func_preproc, abide.phenotypic['DX_GROUP'], abide.phenotypic['SUB_ID']),
        total=len(abide.func_preproc), desc='Extracting FC'):
    label_bin = 1 if int(label) == 1 else 0
    out_path  = os.path.join(FC_DIR, f'{sid}.npy')
    if os.path.exists(out_path):
        records.append({'subject_id': sid, 'label': label_bin, 'fc_path': out_path})
        continue
    try:
        ts = masker.fit_transform(func_path)
        fc = conn.fit_transform([ts])[0]
        np.save(out_path, fc.astype(np.float32))
        records.append({'subject_id': sid, 'label': label_bin, 'fc_path': out_path})
    except Exception as e:
        print(f'  skip {sid}: {e}')

df = pd.DataFrame(records)
df.to_csv(os.path.join(DATA_DIR, 'manifest.csv'), index=False)
print(f'Done! {len(df)} subjects — ASD:{df["label"].sum()}  TC:{(df["label"]==0).sum()}')
print(f'Saved to: {DATA_DIR}')

## Cell 6 — Sanity check: plot a sample FC matrix

In [ ]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd, os
from configs import DATA_DIR, FC_DIR

df  = pd.read_csv(os.path.join(DATA_DIR, 'manifest.csv'))
row = df.iloc[0]
fc  = np.load(row['fc_path'])
print(f'Subject {row["subject_id"]}  label={row["label"]}  fc shape={fc.shape}')

plt.figure(figsize=(7,6))
plt.imshow(fc, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(); plt.title('Sample FC matrix (116 ROIs)'); plt.show()

## Cell 7 — Train BrainIB

In [ ]:
import os, sys, json, time, numpy as np, torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
sys.path.insert(0, '/content/brainib')
os.environ['DRIVE_ROOT'] = '/content/drive/MyDrive'

from configs import CKPT_DIR, RESULTS_DIR, EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY, PATIENCE, BETA, GAMMA, SEED
from data.dataset import ABIDEDataset
from models.brainib import BrainIB
from models.losses import ib_loss
from utils.metrics import compute_metrics, edge_sparsity
from utils.visualize import plot_training_curves

torch.manual_seed(SEED); np.random.seed(SEED)
for d in [CKPT_DIR, RESULTS_DIR]: os.makedirs(d, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

train_ds = ABIDEDataset('train'); val_ds = ABIDEDataset('val')
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)

model     = BrainIB().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_auc = 0.0; no_improve = 0
ckpt_path = os.path.join(CKPT_DIR, 'brainib_best.pt')

for epoch in range(1, EPOCHS+1):
    # ── train ──
    model.train()
    tr_loss = tr_labels = tr_probs = 0
    all_labels, all_probs = [], []
    for batch in train_loader:
        batch = batch.to(device); optimizer.zero_grad()
        out = model(batch)
        loss, *_ = ib_loss(out['logits'], batch.y.squeeze(), out['mu'], out['logvar'], out['edge_mask'])
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        tr_loss += loss.item()
        all_labels.extend(batch.y.squeeze().cpu().numpy())
        all_probs.extend(torch.softmax(out['logits'],dim=-1)[:,1].detach().cpu().numpy())
    tr_m = compute_metrics(all_labels, all_probs)

    # ── val ──
    model.eval(); vl_loss = 0; val_labels=[]; val_probs=[]; hard_masks=[]
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device); out = model(batch)
            l, *_ = ib_loss(out['logits'], batch.y.squeeze(), out['mu'], out['logvar'], out['edge_mask'])
            vl_loss += l.item()
            val_labels.extend(batch.y.squeeze().cpu().numpy())
            val_probs.extend(torch.softmax(out['logits'],dim=-1)[:,1].cpu().numpy())
            hard_masks.append(out['hard_mask'].cpu())
    vl_m = compute_metrics(val_labels, val_probs)
    pruned = edge_sparsity(hard_masks)

    history['train_loss'].append(tr_loss/len(train_loader))
    history['val_loss'].append(vl_loss/len(val_loader))
    history['train_acc'].append(tr_m['accuracy'])
    history['val_acc'].append(vl_m['accuracy'])
    scheduler.step(vl_m['auc_roc'])

    if epoch % 5 == 0 or epoch == 1:
        print(f'Ep {epoch:3d} | tr_loss={tr_loss/len(train_loader):.4f} '
              f'val_acc={vl_m["accuracy"]:.4f} val_auc={vl_m["auc_roc"]:.4f} pruned={pruned:.1%}')

    if vl_m['auc_roc'] > best_auc:
        best_auc = vl_m['auc_roc']; no_improve = 0
        torch.save({'epoch':epoch,'model_state':model.state_dict(),'best_auc':best_auc}, ckpt_path)
        print(f'  ✓ best AUC={best_auc:.4f}')
    else:
        no_improve += 1
        if no_improve >= PATIENCE: print(f'Early stop ep {epoch}'); break

plot_training_curves(history)
with open(os.path.join(RESULTS_DIR,'history.json'),'w') as f: json.dump(history,f)
print(f'Training done. Best val AUC = {best_auc:.4f}')

## Cell 8 — (Optional) Train GCN baseline for comparison

In [ ]:
from models.gcn_baseline import VanillaGCN

baseline = VanillaGCN().to(device)
opt_b = torch.optim.Adam(baseline.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_auc_b = 0.0; no_improve_b = 0
ckpt_b = os.path.join(CKPT_DIR, 'baseline_best.pt')

for epoch in range(1, EPOCHS+1):
    baseline.train(); labels_b=[]; probs_b=[]
    for batch in train_loader:
        batch=batch.to(device); opt_b.zero_grad()
        out=baseline(batch)
        loss=F.cross_entropy(out['logits'], batch.y.squeeze())
        loss.backward(); opt_b.step()
        labels_b.extend(batch.y.squeeze().cpu().numpy())
        probs_b.extend(torch.softmax(out['logits'],dim=-1)[:,1].detach().cpu().numpy())

    baseline.eval(); vl_b=[]; vp_b=[]
    with torch.no_grad():
        for batch in val_loader:
            batch=batch.to(device); out=baseline(batch)
            vl_b.extend(batch.y.squeeze().cpu().numpy())
            vp_b.extend(torch.softmax(out['logits'],dim=-1)[:,1].cpu().numpy())
    vm = compute_metrics(vl_b, vp_b)

    if epoch % 10 == 0:
        print(f'Baseline Ep {epoch:3d} | val_acc={vm["accuracy"]:.4f} auc={vm["auc_roc"]:.4f}')
    if vm['auc_roc'] > best_auc_b:
        best_auc_b = vm['auc_roc']; no_improve_b = 0
        torch.save({'epoch':epoch,'model_state':baseline.state_dict(),'best_auc':best_auc_b}, ckpt_b)
    else:
        no_improve_b += 1
        if no_improve_b >= PATIENCE: print('Early stop'); break

print(f'Baseline best val AUC = {best_auc_b:.4f}')

## Cell 9 — Evaluate on test set + glass-brain plots

In [ ]:
import json
from utils.visualize import plot_glass_brain_subgraph, plot_fc_matrix, aggregate_edge_importance

# Load best checkpoint
ckpt = torch.load(os.path.join(CKPT_DIR, 'brainib_best.pt'), map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded checkpoint epoch={ckpt["epoch"]} best_auc={ckpt["best_auc"]:.4f}')

test_ds     = ABIDEDataset('test')
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)

all_labels=[]; all_probs=[]; hard_masks=[]
outputs_asd=[]; outputs_tc=[]

with torch.no_grad():
    for batch in test_loader:
        batch=batch.to(device); out=model(batch)
        labels=batch.y.squeeze().cpu().numpy()
        probs=torch.softmax(out['logits'],dim=-1)[:,1].cpu().numpy()
        all_labels.extend(labels); all_probs.extend(probs)
        hard_masks.append(out['hard_mask'].cpu())
        for i, lbl in enumerate(labels):
            rec={'edge_mask':out['edge_mask'].cpu(),'edge_index':batch.edge_index.cpu()}
            if hasattr(batch,'fc_raw'): rec['fc_raw']=batch.fc_raw[i].cpu().numpy()
            (outputs_asd if lbl==1 else outputs_tc).append(rec)

from utils.metrics import compute_metrics, edge_sparsity, print_metrics
metrics = compute_metrics(all_labels, all_probs)
metrics['edge_pruned'] = edge_sparsity(hard_masks)
print_metrics(metrics, prefix='TEST')
print(f'[TEST] edges pruned: {metrics["edge_pruned"]:.1%}')

# ── Glass-brain overlays ──────────────────────────────────────────────────
if outputs_asd:
    score_asd = aggregate_edge_importance(outputs_asd)
    path_asd  = plot_glass_brain_subgraph(score_asd, score_asd,
                  title='ASD discriminative subgraph', filename='glass_brain_ASD.png', label=1)
    from IPython.display import Image, display
    display(Image(path_asd))

if outputs_tc:
    score_tc = aggregate_edge_importance(outputs_tc)
    path_tc  = plot_glass_brain_subgraph(score_tc, score_tc,
                  title='TC discriminative subgraph', filename='glass_brain_TC.png', label=0)
    display(Image(path_tc))

# save metrics
def _conv(o):
    import numpy as np
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, dict): return {k:_conv(v) for k,v in o.items()}
    if isinstance(o, list): return [_conv(v) for v in o]
    return o
with open(os.path.join(RESULTS_DIR,'test_metrics.json'),'w') as f:
    json.dump(_conv(metrics), f, indent=2)
print(f'Metrics saved to {RESULTS_DIR}/test_metrics.json')

## Cell 10 — Display training curves

In [ ]:
from IPython.display import Image, display
from configs import FIG_DIR
import os
curve_path = os.path.join(FIG_DIR, 'brainib_training_curves.png')
if os.path.exists(curve_path):
    display(Image(curve_path))
else:
    print('Run training first')

## Cell 11 — Clean up (optional)
Run this ONLY if you want to delete the data from Drive.

In [ ]:
# import shutil
# from configs import DATA_DIR
# shutil.rmtree(DATA_DIR)
# print(f'Deleted {DATA_DIR}')
print('Cleanup cell — uncomment the lines above to delete brainib_data from Drive')